# 01 — Data Privacy & Governance Analysis

**Role:** Governance and Privacy Officer

**NovaCred Credit Application Governance — Team 6**

This notebook:

0. Identifies PII fields in the dataset
1. Pseudonymazation and Anomyzation
2. Data Mapping to GDPR and EU AI Act
4. Proposal of governance control systems


## System Description

NovaCred is an automated credit scoring system that evaluates loan applicants 
and outputs a binary decision (Approved / Rejected).

The system:
- Uses supervised machine learning
- Processes personal and behavioral data
- Produces automated decisions with legal and financial effects
- Operates without mandatory human review in its current state

Under GDPR Art. 22, this qualifies as automated decision-making.
Under the EU AI Act Annex III, credit scoring is classified as a **High-Risk AI System**.


## 0. Identification of PII fields

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Basic configuration for visualizations
%matplotlib inline

In [ ]:
# Loading the cleaned dataset
df = pd.read_csv('../data/cleaned_credit_applications.csv')

# Display first rows to verify gender and decision columns
df.head()

,_id,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,credit_history_months,...,spending_Education,spending_Adult Entertainment,spending_Gambling,dob_parsed,age,annual_income_imputed,dti_flag,email_invalid,timestamp_missing,spending_exceeds_income
0,app_120,Carolyn Martin,NaN,NaN,NaN,Female,NaN,90240.0,103000.0,96,...,0.0,0.0,0.0,NaN,NaN,False,False,True,True,False
1,app_165,Brandon Moore,NaN,NaN,NaN,Male,NaN,10077.0,66000.0,6,...,0.0,0.0,0.0,NaN,NaN,False,False,True,False,False
2,app_075,Margaret Williams,NaN,NaN,NaN,Unknown,NaN,NaN,61000.0,29,...,0.0,0.0,0.0,NaN,NaN,False,False,True,True,False
3,app_350,Linda Adams,NaN,356-98-8263,10.207.183.196,Female,NaN,90291.0,89000.0,52,...,0.0,0.0,0.0,NaN,NaN,False,False,True,True,False
4,app_437,Ashley Lopez,ashley.lopez68@icloud.com,637-73-3119,192.168.220.15,Female,1992-02-01,10033.0,107000.0,59,...,0.0,0.0,0.0,NaN,NaN,False,False,False,True,False


Feature Categorization for Governance

In [3]:
# Categorize features for governance assessment

direct_identifiers = ['name', 'email', 'ssn']
quasi_identifiers = ['zip_code', 'ip_address']
protected_attributes = ['gender', 'age']
behavioral_features = ['credit_utilization', 'transaction_volume']
target_variable = ['loan_approved']

print("Direct Identifiers:", direct_identifiers)
print("Quasi Identifiers:", quasi_identifiers)
print("Protected Attributes:", protected_attributes)
print("Behavioral Features:", behavioral_features)

Direct Identifiers: ['name', 'email', 'ssn']
Quasi Identifiers: ['zip_code', 'ip_address']
Protected Attributes: ['gender', 'age']
Behavioral Features: ['credit_utilization', 'transaction_volume']


## 0.1 Data Categories Processed

The dataset contains the following governance-relevant feature groups:

### Direct Identifiers
- Name
- Email
- SSN

### Quasi-Identifiers
- ZIP code
- IP address

### Protected Attributes
- Gender
- Age

### Behavioral / Financial Data
- Credit utilization
- Transaction history
- Debt-to-income ratio

### Target Variable
- Loan approval decision

All above categories qualify as personal data under GDPR Art. 4(1).

## 0.2 Scope of Governance Review

This governance audit covers the following system components:


1. **Training Data Pipeline**
   - Data ingestion
   - Feature engineering
   - Storage of raw and processed data

2. **Inference & Decision Pipeline**
   - Model scoring
   - Threshold application
   - Automated approval/rejection decision

3. **Logging & Record-Keeping**
   - Decision traceability
   - Model version logging
   - Data retention policies

4. **Human Oversight Process**
   - Existence of manual review mechanisms
   - Override documentation
   - Bias monitoring procedures

5. **Consent & Legal Basis**
   - Consent collection mechanisms
   - Lawful basis documentation
   - Right to erasure implementation


## 1. Pseudonymization & Anonymization Demonstration (Governance Controls)

To reduce privacy risk while maintaining auditability:

- **Gender (protected attribute):** Pseudonymized via controlled tokenization.
  This reduces exposure of the raw attribute while preserving the ability
  to conduct fairness audits under restricted access.

- **Age (quasi-identifier):** Generalized into predefined age groups.
  Exact age and date of birth are removed to reduce re-identification risk.

### Gender Pseudonymization (GDPR Art. 4(5))

A mapping table replaces raw gender values with neutral tokens.
The mapping table constitutes "additional information" and must be
stored separately with access controls.

This aligns with GDPR Art. 4(5), as attribution requires access
to the separately stored mapping table.

In [29]:
# Step 1: Define controlled token mapping
gender_mapping = {
    "Female": "G1",
    "Male": "G2",
    "Unknown": "G3"
}

# Step 2: Apply pseudonymization with fallback
df["gender_token"] = df["gender"].map(gender_mapping).fillna("G_UNKNOWN")

# Step 3: Simulate separate storage of mapping table
gender_lookup_table = pd.DataFrame(
    list(gender_mapping.items()),
    columns=["original_gender", "gender_token"]
)

# Step 4: Remove raw gender from working dataset
df_anonymized = df.drop(columns=["gender"]).copy()

df[["gender", "gender_token"]].head()

,gender,gender_token
0,Female,G1
1,Male,G2
2,Unknown,G3
3,Female,G1
4,Female,G1


In [ ]:
# Create mapping table as separate object
gender_lookup_table = pd.DataFrame(
    list(gender_mapping.items()),
    columns=["original_gender", "gender_token"]
)

gender_lookup_table

,original_gender,gender_token
0,Female,G1
1,Male,G2
2,Unknown,G3


### Justification of Technique Choice

Among the three pseudonymization approaches discussed in class:

- **Tokenization** was selected over hashing and encryption.
- Unlike hashing, tokenization does not create a deterministic mathematical link.
- Unlike encryption, operational datasets do not require decryption keys.

While tokenization introduces the risk of lookup-table compromise,
it provides strong separation between operational data and protected attributes,
which aligns with privacy-by-design principles.

### Governance Note

In a production environment, the mapping table would be stored in a secure,
access-controlled repository (e.g., encrypted database or vault).

Operational datasets (e.g., `df_anonymized`) would only contain `gender_token`,
while the lookup table would be accessible exclusively to authorized governance
or compliance personnel.

This separation implements a technical and organizational control,
ensuring that the semantic meaning of the token cannot be restored
without access to the separately stored mapping table.

### Compliance Justification

The pseudonymization of gender reduces the exposure of protected attributes
during routine data processing while preserving the ability to conduct
fairness audits under controlled conditions.

This implementation aligns with:

- **GDPR Art. 4(5)** – Definition of pseudonymization (processing in such a manner that attribution requires additional information kept separately).
- **GDPR Art. 5(1)(c)** – Data minimization (limiting exposure of protected attributes).
- **GDPR Art. 32** – Security of processing (implementation of appropriate technical and organizational measures).

By separating the operational dataset from the mapping table,
the risk of unauthorized attribute disclosure is reduced while
maintaining necessary auditability for fairness monitoring.

## 1.2 Age Anonymization (Generalization)

Exact age is a quasi-identifier and increases re-identification risk.  
To anonymize this field, we replace the exact numeric value (`age`) with **age group** (`age_group`).

This approach reduces identifiability through generalization and supports anonymization objectives:

- Removes exact age values from the working dataset
- Retains analytical utility (e.g., fairness checks for 18-25 year old applicants)
- Supports data minimization / privacy-by-design principles

While generalization reduces re-identification risk, it does not guarantee absolute anonymization and therefore, small-group suppression may be applied
to further mitigate uniqueness risk.

In [24]:
# Parse DOB safely
df["dob_parsed"] = pd.to_datetime(df["date_of_birth"], errors="coerce")

# Calculate precise age
today = pd.Timestamp.today()
df["age"] = (today - df["dob_parsed"]).dt.days // 365

print("Missing age after recalculation:", df["age"].isna().sum())

# Recreate original age groups for consistency with bias analysis
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 25, 40, 60, 100],
    labels=["18-25", "26-40", "41-60", "60+"],
    include_lowest=True
)

print("\nAge group distribution:")
print(df["age_group"].value_counts(dropna=False))


Missing age after recalculation: 4

Age group distribution:
age_group
26-40    250
41-60    185
60+       39
18-25     22
NaN        4
Name: count, dtype: int64


In [26]:
# Creating anonymized dataset 
df_anonymized = df.drop(columns=["age", "dob_parsed", "date_of_birth"])

print("Columns in anonymized dataset:")
print(df_anonymized.columns)

df_anonymized[["age_group"]].head()

Columns in anonymized dataset:
Index(['_id', 'full_name', 'email', 'ssn', 'ip_address', 'gender', 'zip_code',
       'annual_income', 'credit_history_months', 'debt_to_income',
       'savings_balance', 'spending_Shopping', 'spending_Rent',
       'spending_Alcohol', 'spending_total', 'loan_approved', 'interest_rate',
       'approved_amount', 'rejection_reason', 'processing_timestamp',
       'spending_Dining', 'spending_Healthcare', 'spending_Fitness',
       'spending_Entertainment', 'spending_Insurance', 'spending_Travel',
       'spending_Transportation', 'spending_Utilities', 'spending_Groceries',
       'spending_Education', 'spending_Adult Entertainment',
       'spending_Gambling', 'annual_income_imputed', 'dti_flag',
       'email_invalid', 'timestamp_missing', 'spending_exceeds_income',
       'gender_token', 'age_band', 'age_group', 'age_group_anon'],
      dtype='object')


,age_group
0,NaN
1,NaN
2,NaN
3,NaN
4,26-40


In [28]:
k = 10

# Apply suppression to anonymized dataset
counts = df_anonymized["age_group"].value_counts()

df_anonymized["age_group_anon"] = df_anonymized["age_group"].astype("object")

small_groups = counts[counts < k].index

df_anonymized.loc[
    df_anonymized["age_group"].isin(small_groups),
    "age_group_anon"
] = "Other/SmallGroup"

print("Age group distribution after suppression:")
print(df_anonymized["age_group_anon"].value_counts(dropna=False))

df_anonymized = df_anonymized.drop(columns=["age_group"])

Age group distribution after suppression:
age_group_anon
26-40    250
41-60    185
60+       39
18-25     22
NaN        4
Name: count, dtype: int64


### Governance Note

- The original `age` and `date_of_birth` columns are removed from `df_anonymized`.
- Only generalized age categories (`age_group`) are retained.
- Where applicable, small groups are suppressed using a k-threshold (k=10) 
  to reduce uniqueness risk.

These measures reduce re-identification risk while preserving analytical utility for group-level fairness monitoring.

### Quasi-identifier Risk

Even if direct identifiers (name, email, SSN) are removed, 
quasi-identifiers such as ZIP code, gender, and date of birth 
can enable re-identification when combined.

Therefore:
- `gender` is pseudonymized via controlled tokenization.
- Exact age and date of birth are not exposed in the analytics dataset.
- Age is generalized into predefined categories.
- Small groups may be suppressed (k-anonymity inspired control) to further mitigate uniqueness risk.

## 2. Regulatory Mapping: GDPR & EU AI Act

This section maps the NovaCred credit scoring system to applicable legal obligations under:

- The General Data Protection Regulation (GDPR)
- The EU AI Act (High-Risk AI Systems)

The objective is to demonstrate regulatory awareness and identify
required governance controls.

### 2.1 Identified Bias Types

| Bias Type | Evidence in NovaCred | Governance Intervention |
|------------|----------------------|--------------------------|
| Historical Bias | Gender approval disparity (DI < 0.8) | Recalibration + fairness constraints |
| Selection Bias | Potential underrepresentation of young applicants | Improve sampling & monitor demographic balance |
| Measurement Bias | ZIP code used as proxy for creditworthiness | Remove proxy variables |
| Aggregation Bias | Single model applied to all age groups | Intersectional fairness monitoring |
| Feedback Loop Bias | Rejected applicants generate less future data | Continuous monitoring & periodic retraining |

### 2.1 GDPR Applicability

As mentioned before NovaCred processes personal data including:
- Direct identifiers (name, email, SSN)
- Quasi-identifiers (ZIP code, date of birth)
- Protected attributes (gender)
- Financial and behavioral data

As the system performs automated credit decisions with legal effects, GDPR Art. 22 (Automated Individual Decision-Making) is triggered.

### 2.2 GDPR Article Mapping

| GDPR Article | Requirement | NovaCred Risk | Recommended Control |
|--------------|------------|--------------|------------------------------------|
| Art. 4(5) | Pseudonymization definition | Gender exposed in raw form | Tokenization with separate mapping table |
| Art. 5(1)(c) | Data minimization | Exact DOB & age stored | Age generalization + removal of raw DOB |
| Art. 5(1)(e) | Storage limitation | No retention policy defined | Define retention schedule |
| Art. 6 | Lawful basis | Consent tracking unclear | Implement consent registry |
| Art. 22 | Automated decision-making | Fully automated credit approval | Introduce human oversight mechanism |
| Art. 32 | Security of processing | Sensitive data stored in plaintext | Encryption + access controls |
| Art. 35 | DPIA required | High-risk profiling system | Conduct Data Protection Impact Assessment |

### 2.3 EU AI Act Classification

In the Annex III of the EU AI Act, AI systems used for credit scoring and access to financial services are classified as **High-Risk AI Systems**.

Therefore, NovaCred must comply with multiple governance, risk management, and documentation requirements.

### 2.4 EU AI Act Obligation Mapping

| EU AI Act Obligation | Relevance to NovaCred | Governance Status |
|----------------------|-----------------------|-------------------|
| Risk Management System (Art. 9) | Bias and discrimination risk | Bias audit conducted |
| Data Governance (Art. 10) | Training data quality & representativeness | Feature review + proxy removal |
| Technical Documentation (Art. 11) | Transparency & auditability | Model documentation recommended |
| Record-Keeping (Art. 12) | Decision traceability | Audit trail needed |
| Transparency (Art. 13) | Inform users of automated decision | Notice required |
| Human Oversight (Art. 14) | Fully automated approval process | Manual review mechanism required |
| Accuracy & Robustness (Art. 15) | Model fairness + reliability | Ongoing monitoring required |

### 2.5 Regulatory Risk Summary

NovaCred currently presents regulatory gaps in:

- Consent management
- Decision logging
- Human oversight documentation
- Data retention policy
- Fairness monitoring

While technical bias mitigation measures were proposed, full compliance requires implementation of organizational and procedural governance controls.